# TidalSense N-Tidal Clinical Study
## Part 1: Study Health Report

This notebook covers the operational summary of the 30-day clinical study.
It looks at test volumes, error rates, diagnostic outcomes, and any notable patterns.

Data sources:
- software_table.csv: one row per test
- hardware_telemetry_table.csv: periodic device logs

## Step 1: Load the Data

In [ ]:
import pandas as pd

# Load both tables
sw = pd.read_csv('software_table.csv')
hw = pd.read_csv('hardware_telemetry_table.csv')

# Convert timestamp columns to datetime so we can do date calculations
sw['test_log_time'] = pd.to_datetime(sw['test_log_time'])
sw['test_processed_time'] = pd.to_datetime(sw['test_processed_time'])
hw['log_time'] = pd.to_datetime(hw['log_time'])

print('Software table shape:', sw.shape)
print('Hardware table shape:', hw.shape)

In [ ]:
# Quick look at the columns and first few rows
print('Software table columns:')
print(sw.columns.tolist())
print()
sw.head(3)

In [ ]:
print('Hardware table columns:')
print(hw.columns.tolist())
print()
hw.head(3)

## Step 2: Add Helper Columns

We create a few extra columns that will be useful throughout the analysis.

In [ ]:
# Extract just the date part from the timestamp (we will use this to count tests per day)
sw['date'] = sw['test_log_time'].dt.date

# Get the day of week name so we can separate weekdays from weekends
sw['weekday'] = sw['test_log_time'].dt.day_name()

# True/False column: is this test on a weekend?
sw['is_weekend'] = sw['weekday'].isin(['Saturday', 'Sunday'])

# True/False column: did this test have any kind of error?
# We combine all four error columns into one flag
sw['any_error'] = (
    sw['automated_check_hardware_error'] |
    sw['automated_check_software_error'] |
    (sw['user_error'] != 'no_error') |
    sw['miscalibration']
)

# Simpler version of the user_error column: True if there was any user error
sw['has_user_error'] = sw['user_error'] != 'no_error'

print('New columns added: date, weekday, is_weekend, any_error, has_user_error')

## Section 1: Overall Study Summary

In [ ]:
# Top level numbers for the study
total_tests = len(sw)
num_practices = sw['practice_ID'].nunique()
num_handsets = sw['handset_ID'].nunique()
study_start = sw['test_log_time'].min().date()
study_end = sw['test_log_time'].max().date()

print('Total tests logged:  ', total_tests)
print('Practices:           ', num_practices)
print('Handsets:            ', num_handsets)
print('Study start:         ', study_start)
print('Study end:           ', study_end)

## Section 2: Test Volumes

In [ ]:
# Count how many tests each practice ran
tests_per_practice = sw.groupby('practice_ID').size().sort_values(ascending=False)
tests_per_practice = tests_per_practice.rename('test_count')

print('Tests per practice:')
print(tests_per_practice)

In [ ]:
# Count how many tests each handset ran
tests_per_handset = sw.groupby('handset_ID').size().sort_values(ascending=False)
tests_per_handset = tests_per_handset.rename('test_count')

print('Tests per handset:')
print(tests_per_handset)

In [ ]:
# Count tests per day to see if volumes are stable or fluctuating
tests_per_day = sw.groupby('date').size().rename('test_count')

print('Tests per day:')
print(tests_per_day.to_string())

In [ ]:
# Separate the daily counts into weekday vs weekend to compare averages
weekday_tests = sw[sw['is_weekend'] == False]
weekend_tests = sw[sw['is_weekend'] == True]

weekday_daily_avg = weekday_tests.groupby('date').size().mean()
weekend_daily_avg = weekend_tests.groupby('date').size().mean()

print('Average tests per weekday: ', round(weekday_daily_avg, 1))
print('Average tests per weekend day: ', round(weekend_daily_avg, 1))
print()
print('Weekend tests by practice:')
print(weekend_tests.groupby('practice_ID').size())

**Observation:** All weekend tests belong to PRC-009 only. No other practice ran tests on weekends. This is unusual and flagged in Part 2 as a protocol issue.

## Section 3: Diagnostic Outcomes

In [ ]:
# Overall split between positive and negative results
diagnosis_counts = sw['diagnosis'].value_counts()
diagnosis_pct = sw['diagnosis'].value_counts(normalize=True).mul(100).round(1)

print('Diagnosis counts:')
print(diagnosis_counts)
print()
print('Diagnosis percentages:')
print(diagnosis_pct)

In [ ]:
# Positive rate broken down by practice
# We group by practice and calculate what fraction of tests returned positive
positive_rate_by_practice = sw.groupby('practice_ID')['diagnosis'].apply(
    lambda x: round((x == 'positive').mean() * 100, 1)
).sort_values(ascending=False)

positive_rate_by_practice = positive_rate_by_practice.rename('positive_rate_pct')

print('Positive rate by practice (%):')
print(positive_rate_by_practice)

## Section 4: Error Rates

In [ ]:
# Overall rates for each error type across all tests
hw_error_rate = sw['automated_check_hardware_error'].mean() * 100
sw_error_rate = sw['automated_check_software_error'].mean() * 100
user_error_rate = sw['has_user_error'].mean() * 100
miscal_rate = sw['miscalibration'].mean() * 100
any_error_rate = sw['any_error'].mean() * 100

print('Error rates (% of all tests):')
print('  Any error:        ', round(any_error_rate, 1), '%')
print('  Hardware error:   ', round(hw_error_rate, 1), '%')
print('  Software error:   ', round(sw_error_rate, 1), '%')
print('  User error:       ', round(user_error_rate, 1), '%')
print('  Miscalibration:   ', round(miscal_rate, 1), '%')

In [ ]:
# Break down the user error type further
print('User error breakdown:')
print(sw['user_error'].value_counts())

In [ ]:
# Overall error rate per practice (any error flag)
# This tells us if any practice stands out as having more problems than others
error_by_practice = sw.groupby('practice_ID')['any_error'].mean().mul(100).round(1)
error_by_practice = error_by_practice.sort_values(ascending=False).rename('any_error_rate_pct')

print('Overall error rate by practice (%):')
print(error_by_practice)

In [ ]:
# Miscalibration rate by practice separately, because it is the biggest signal
miscal_by_practice = sw.groupby('practice_ID')['miscalibration'].mean().mul(100).round(1)
miscal_by_practice = miscal_by_practice.sort_values(ascending=False).rename('miscalibration_rate_pct')

print('Miscalibration rate by practice (%):')
print(miscal_by_practice)

**Observation:** PRC-007 has a miscalibration rate of 30.5% against a study average of around 1.7% for all other practices. This is investigated in depth in Part 2.

## Section 5: Summary Table

In [ ]:
# Build one summary table per practice combining volume, error rate, and positive rate
# Each calculation is done separately and then joined together

vol = sw.groupby('practice_ID').size().rename('total_tests')

pos_rate = sw.groupby('practice_ID')['diagnosis'].apply(
    lambda x: round((x == 'positive').mean() * 100, 1)
).rename('positive_rate_pct')

err_rate = sw.groupby('practice_ID')['any_error'].mean().mul(100).round(1).rename('any_error_rate_pct')

miscal = sw.groupby('practice_ID')['miscalibration'].mean().mul(100).round(1).rename('miscalibration_pct')

# Join all four series into one dataframe
summary = pd.concat([vol, pos_rate, err_rate, miscal], axis=1)
summary = summary.sort_values('any_error_rate_pct', ascending=False)

print('Practice-level summary:')
print(summary.to_string())